In [1]:
# ==========================================================
# DenseNet121 HEp-2 Pattern Classification
# ==========================================================
# Purpose:
# Train DenseNet121 for six-class HEp-2 staining pattern
# classification.
#
# Objective Connection:
# Objective 1 - Automated HEp-2 pattern recognition
# Objective 3 - Comparison of deep learning models
# ==========================================================

import os
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

PyTorch version: 2.13.0+cpu
Torchvision version: 0.28.0+cpu
Device: cpu


In [2]:
# ==========================================================
# Dataset Path
# ==========================================================

DATASET_PATH = "hep2_data/data"

print("Dataset path:", DATASET_PATH)
print("Classes:", sorted(os.listdir(DATASET_PATH)))

Dataset path: hep2_data/data
Classes: ['Centromere', 'Golgi', 'Homogeneous', 'NuMem', 'Nucleolar', 'Speckled']


In [3]:
# ==========================================================
# Stratified Train / Validation / Test Split
# ----------------------------------------------------------
# Same split used for ResNet50 for a fair comparison.
# 70% Training | 15% Validation | 15% Testing
# ==========================================================

from sklearn.model_selection import train_test_split

# Collect image paths and labels
image_paths = []
labels = []

for class_name in sorted(os.listdir(DATASET_PATH)):
    class_folder = os.path.join(DATASET_PATH, class_name)

    if os.path.isdir(class_folder):
        for filename in os.listdir(class_folder):
            if filename.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
                image_paths.append(os.path.join(class_folder, filename))
                labels.append(class_name)

df = pd.DataFrame({
    "image_path": image_paths,
    "label": labels
})

print("Total images:", len(df))

# Same random seed as ResNet
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("\nDataset split completed.")
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTesting distribution:")
print(test_df["label"].value_counts().sort_index())

Total images: 13596

Dataset split completed.
Training images: 9517
Validation images: 2039
Testing images: 2040

Training distribution:
label
Centromere     1919
Golgi           507
Homogeneous    1746
NuMem          1545
Nucleolar      1818
Speckled       1982
Name: count, dtype: int64

Validation distribution:
label
Centromere     411
Golgi          109
Homogeneous    374
NuMem          331
Nucleolar      390
Speckled       424
Name: count, dtype: int64

Testing distribution:
label
Centromere     411
Golgi          108
Homogeneous    374
NuMem          332
Nucleolar      390
Speckled       425
Name: count, dtype: int64


In [4]:
# ==========================================================
# DenseNet121 Image Transformations
# ----------------------------------------------------------
# Use the same preprocessing strategy as ResNet50 so that
# both models are compared fairly.
# ==========================================================

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transformations defined successfully.")

Transformations defined successfully.


In [5]:
# ==========================================================
# HEp-2 Dataset Class
# ----------------------------------------------------------
# Creates a PyTorch Dataset using the same train/validation/
# test split prepared above.
# ==========================================================

from PIL import Image
from torch.utils.data import Dataset, DataLoader

class HEp2Dataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

        self.class_names = sorted(self.dataframe["label"].unique())
        self.class_to_idx = {
            name: idx for idx, name in enumerate(self.class_names)
        }

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(row["image_path"]).convert("RGB")
        label = self.class_to_idx[row["label"]]

        if self.transform:
            image = self.transform(image)

        return image, label


# Create datasets
train_dataset = HEp2Dataset(
    train_df,
    transform=train_transform
)

val_dataset = HEp2Dataset(
    val_df,
    transform=val_test_transform
)

test_dataset = HEp2Dataset(
    test_df,
    transform=val_test_transform
)

class_names = sorted(df["label"].unique())

print("Classes:", class_names)
print("Number of classes:", len(class_names))
print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Classes: ['Centromere', 'Golgi', 'Homogeneous', 'NuMem', 'Nucleolar', 'Speckled']
Number of classes: 6
Train dataset: 9517
Validation dataset: 2039
Test dataset: 2040


In [6]:
# ==========================================================
# DenseNet121 DataLoaders
# ==========================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully.")
print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

DataLoaders created successfully.
Batch size: 32
Training batches: 298
Validation batches: 64
Testing batches: 64


In [7]:
# ==========================================================
# DenseNet121 Model Setup
# ----------------------------------------------------------
# Load ImageNet-pretrained DenseNet121 and replace the final
# classifier for our 6 HEp-2 staining classes.
# ==========================================================

from torchvision.models import densenet121, DenseNet121_Weights

# Load pretrained DenseNet121
weights = DenseNet121_Weights.DEFAULT

model = densenet121(weights=weights)

# Replace the final classifier
num_features = model.classifier.in_features

model.classifier = nn.Linear(
    num_features,
    len(class_names)
)

# Move model to CPU/GPU
model = model.to(device)

print("DenseNet121 loaded successfully.")
print("Original classifier input features:", num_features)
print("Number of output classes:", len(class_names))
print("Device:", device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\Sinchana Raj/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:02<00:00, 12.6MB/s]


DenseNet121 loaded successfully.
Original classifier input features: 1024
Number of output classes: 6
Device: cpu


In [8]:
# ==========================================================
# DenseNet121 Training Configuration
# ==========================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

num_epochs = 10

print("Loss function:", criterion)
print("Optimizer: Adam")
print("Learning rate:", 0.0001)
print("Epochs:", num_epochs)

Loss function: CrossEntropyLoss()
Optimizer: Adam
Learning rate: 0.0001
Epochs: 10


In [9]:
# ==========================================================
# DenseNet121 Training
# ==========================================================

best_val_loss = float("inf")
best_model_state = None

print("=" * 60)
print("DENSENET121 TRAINING STARTED")
print("=" * 60)
print("Device:", device)
print("Epochs:", num_epochs)
print()

for epoch in range(num_epochs):

    epoch_start = time.time()

    # -------------------------
    # Training
    # -------------------------
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predictions = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

        # Progress every 20 batches
        if batch_idx % 20 == 0:
            print(
                f"[Epoch {epoch + 1}] "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    train_loss = running_loss / total
    train_acc = correct / total

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, predictions = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predictions == labels).sum().item()

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    epoch_time = (time.time() - epoch_start) / 60

    # -------------------------
    # Save best model
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(model.state_dict())

        torch.save(
            best_model_state,
            "densenet121_best.pth"
        )

        best_text = " <-- BEST MODEL SAVED"

    else:
        best_text = ""

    print()
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Time: {epoch_time:.1f} min"
        f"{best_text}"
    )

    print("-" * 60)

print()
print("=" * 60)
print("DENSENET121 TRAINING COMPLETED")
print("=" * 60)
print(f"Best validation loss: {best_val_loss:.6f}")
print("Best model saved as: densenet121_best.pth")

DENSENET121 TRAINING STARTED
Device: cpu
Epochs: 10

[Epoch 1] Batch 0/298 | Loss: 1.9564
[Epoch 1] Batch 20/298 | Loss: 0.6380
[Epoch 1] Batch 40/298 | Loss: 0.8088
[Epoch 1] Batch 60/298 | Loss: 0.8351
[Epoch 1] Batch 80/298 | Loss: 0.4691
[Epoch 1] Batch 100/298 | Loss: 0.2574
[Epoch 1] Batch 120/298 | Loss: 0.1242
[Epoch 1] Batch 140/298 | Loss: 0.1698
[Epoch 1] Batch 160/298 | Loss: 0.0923
[Epoch 1] Batch 180/298 | Loss: 0.2983
[Epoch 1] Batch 200/298 | Loss: 0.1637
[Epoch 1] Batch 220/298 | Loss: 0.1796
[Epoch 1] Batch 240/298 | Loss: 0.3084
[Epoch 1] Batch 260/298 | Loss: 0.1824
[Epoch 1] Batch 280/298 | Loss: 0.3359

Epoch [1/10] | Train Loss: 0.3858 | Train Acc: 0.8688 | Val Loss: 0.1739 | Val Acc: 0.9397 | Time: 105.5 min <-- BEST MODEL SAVED
------------------------------------------------------------
[Epoch 2] Batch 0/298 | Loss: 0.1376
[Epoch 2] Batch 20/298 | Loss: 0.2634
[Epoch 2] Batch 40/298 | Loss: 0.2630
[Epoch 2] Batch 60/298 | Loss: 0.0839
[Epoch 2] Batch 80/298 | 

In [10]:
# ==========================================================
# DenseNet121 Test Evaluation
# ==========================================================

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Load the BEST DenseNet121 model
best_model_path = "densenet121_best.pth"

model.load_state_dict(torch.load(
    best_model_path,
    map_location=device
))

model = model.to(device)
model.eval()

all_predictions = []
all_labels = []

test_loss = 0.0
correct = 0
total = 0

criterion = torch.nn.CrossEntropyLoss()

print("=" * 60)
print("DENSENET121 TEST EVALUATION")
print("=" * 60)

with torch.no_grad():

    for batch_idx, (images, labels) in enumerate(test_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        _, predictions = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 20 == 0:
            print(f"Test Batch {batch_idx}/{len(test_loader)}")

# Calculate metrics
test_loss = test_loss / total
test_accuracy = correct / total

precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 60)
print("FINAL DENSENET121 TEST RESULTS")
print("=" * 60)

print(f"Test Loss      : {test_loss:.4f}")
print(f"Test Accuracy  : {test_accuracy:.4f}")
print(f"Test Accuracy  : {test_accuracy * 100:.2f}%")
print(f"Precision      : {precision:.4f}")
print(f"Recall         : {recall:.4f}")
print(f"F1 Score       : {f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        zero_division=0
    )
)

DENSENET121 TEST EVALUATION
Test Batch 0/64
Test Batch 20/64
Test Batch 40/64
Test Batch 60/64

FINAL DENSENET121 TEST RESULTS
Test Loss      : 0.0856
Test Accuracy  : 0.9755
Test Accuracy  : 97.55%
Precision      : 0.9755
Recall         : 0.9755
F1 Score       : 0.9754

Classification Report:
              precision    recall  f1-score   support

  Centromere       0.98      0.99      0.99       411
       Golgi       0.97      0.94      0.96       108
 Homogeneous       0.97      0.99      0.98       374
       NuMem       0.98      0.98      0.98       332
   Nucleolar       0.98      0.97      0.97       390
    Speckled       0.97      0.96      0.96       425

    accuracy                           0.98      2040
   macro avg       0.98      0.97      0.97      2040
weighted avg       0.98      0.98      0.98      2040

